# OMIn GraphRAG 6-Guardrails (+Expanded Benchmark)

**Two guardrails:**

### Pre-Generation Guardrail
Inspired by NeMo Guardrails (Rebedea et al., 2023) canonical form extraction.
Before generating an answer:
1. Extract canonical form of query (structured intent representation)
2. Generate ideal subschema — what entity types and relation types SHOULD be present
3. Compare ideal subschema vs retrieved subgraph (structural + semantic match)
4. If match score below threshold: flag low confidence, expand retrieval, or abstain
5. Log retrieval trace: 'searched X, found Y, confidence Z'

### Post-Generation Guardrail  
After generating an answer:
1. NLI grounding check: does the answer entail from retrieved records?
2. Graph trace attribution: every claim in answer should trace to a retrieved edge
3. Flag hallucinated content not grounded in retrieved records



## Setup & load all artifacts

In [ ]:
!pip install -q google-genai networkx numpy scikit-learn tqdm

In [ ]:
import json, os, time
import numpy as np
import networkx as nx
from collections import defaultdict
from tqdm.notebook import tqdm
from google import genai

GEMINI_API_KEY   = ""
client           = genai.Client(api_key=GEMINI_API_KEY)
GENERATION_MODEL = "gemini-2.5-flash"
EMBEDDING_MODEL  = "gemini-embedding-001"

# Load all artifacts from previous notebooks
with open("./output/knowledge_graph.json") as f:
    graph_json = json.load(f)
with open("./output/canonical_embeddings.json") as f:
    entity_embeddings = json.load(f)
with open("./output/relation_embeddings.json") as f:
    relation_embeddings = json.load(f)
with open("./output/records.json") as f:
    records = json.load(f)
with open("./output/graph_schema.json") as f:
    graph_schema = json.load(f)
with open("./output/qa_benchmark.json") as f:
    qa_benchmark = json.load(f)
with open("./output/eval_results_baseline.json") as f:
    baseline_results = json.load(f)

record_lookup = {r["record_id"]: r for r in records}

# Rebuild graph
G = nx.MultiDiGraph()
for node in graph_json["nodes"]:
    G.add_node(node["id"], record_ids=node["record_ids"], frequency=node["frequency"])
for edge in graph_json["edges"]:
    G.add_edge(edge["head"], edge["tail"],
               relation=edge["relation"], weight=edge["weight"],
               record_ids=edge["record_ids"], support_count=edge["support_count"])

# Build entity embedding matrix
entity_list   = list(entity_embeddings.keys())
entity_matrix = np.array([entity_embeddings[e] for e in entity_list], dtype=np.float32)
norms         = np.linalg.norm(entity_matrix, axis=1, keepdims=True)
entity_matrix_norm = entity_matrix / (norms + 1e-9)
entity_to_idx = {e: i for i, e in enumerate(entity_list)}

RELATION_TYPES = [
    "CAUSES", "RESULTS_IN", "PART_OF", "LOCATED_IN",
    "REQUIRES_ACTION", "INSPECTED_BY", "REPLACED_BY",
    "PRECEDED_BY", "AFFECTS", "INSTANCE_OF"
]

print(f"All artifacts loaded.")
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

## Copy pipeline functions from notebook 4
All functions needed for the full pipeline.

In [ ]:
#debug 
import re

def safe_parse_json(raw):
    """Try multiple strategies to parse potentially malformed JSON from LLM."""
    # Strategy 1: direct parse
    try:
        return json.loads(raw)
    except:
        pass
    # Strategy 2: find JSON object with regex
    try:
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        if match:
            return json.loads(match.group())
    except:
        pass
    # Strategy 3: fix common issues (trailing commas, single quotes)
    try:
        fixed = re.sub(r',\s*}', '}', raw)
        fixed = re.sub(r',\s*]', ']', fixed)
        return json.loads(fixed)
    except:
        pass
    return None

In [ ]:
#All pipeline functions from notebook 4

def embed_query(query, client, model=EMBEDDING_MODEL):
    result = client.models.embed_content(model=model, contents=[query])
    emb = np.array(result.embeddings[0].values, dtype=np.float32)
    return emb / (np.linalg.norm(emb) + 1e-9)

def find_seed_nodes(query_emb, entity_matrix_norm, entity_list, G,
                    top_k=10, diversity_lambda=0.5):
    similarities = entity_matrix_norm @ query_emb
    candidates = [(entity_list[i], float(similarities[i]))
                  for i in range(len(entity_list)) if entity_list[i] in G.nodes]
    candidates.sort(key=lambda x: -x[1])
    candidates = candidates[:200]
    selected, selected_embs = [], []
    while len(selected) < top_k and candidates:
        best_score, best_idx = -999, 0
        for i, (entity, sim_to_query) in enumerate(candidates):
            if not selected_embs:
                mmr_score = sim_to_query
            else:
                entity_emb = entity_matrix_norm[entity_to_idx[entity]]
                max_sim = max(float(np.dot(entity_emb, s)) for s in selected_embs)
                mmr_score = diversity_lambda * sim_to_query - (1 - diversity_lambda) * max_sim
            if mmr_score > best_score:
                best_score, best_idx = mmr_score, i
        entity, sim = candidates.pop(best_idx)
        selected.append({"node": entity, "similarity": sim, "mmr_score": best_score})
        selected_embs.append(entity_matrix_norm[entity_to_idx[entity]])
    return selected

def expand_subgraph(seed_nodes, G, hops=2):
    visited  = set(s["node"] for s in seed_nodes)
    frontier = set(visited)
    for _ in range(hops):
        new_frontier = set()
        for node in frontier:
            if node not in G: continue
            for nb in list(G.successors(node)) + list(G.predecessors(node)):
                if nb not in visited:
                    new_frontier.add(nb)
                    visited.add(nb)
        frontier = new_frontier
    subgraph_edges = []
    for u, v, data in G.edges(data=True):
        if u in visited and v in visited:
            subgraph_edges.append({"head": u, "tail": v,
                "relation": data["relation"], "record_ids": data["record_ids"],
                "support_count": data["support_count"]})
    return subgraph_edges, visited

def score_edges_semantic(subgraph_edges, query_emb, entity_embeddings,
                          relation_embeddings, top_n=30):
    scored = []
    for edge in subgraph_edges:
        head_emb = entity_embeddings.get(edge["head"])
        tail_emb = entity_embeddings.get(edge["tail"])
        rel_emb  = relation_embeddings.get(edge["relation"])
        if head_emb is None or tail_emb is None or rel_emb is None: continue
        h = np.array(head_emb, dtype=np.float32); h /= (np.linalg.norm(h) + 1e-9)
        t = np.array(tail_emb, dtype=np.float32); t /= (np.linalg.norm(t) + 1e-9)
        r = np.array(rel_emb,  dtype=np.float32); r /= (np.linalg.norm(r)  + 1e-9)
        score = 0.4 * float(np.dot(query_emb, h)) + \
                0.4 * float(np.dot(query_emb, t)) + \
                0.2 * float(np.dot(query_emb, r))
        scored.append({**edge, "relevance_score": score})
    scored.sort(key=lambda x: -x["relevance_score"])
    return scored[:top_n]

def retrieve_records(scored_edges, record_lookup, max_records=15):
    record_scores   = defaultdict(float)
    record_edge_map = defaultdict(list)
    for edge in scored_edges:
        for rid in edge["record_ids"]:
            record_scores[rid] += edge["relevance_score"]
            record_edge_map[rid].append({"head": edge["head"], "relation": edge["relation"],
                                          "tail": edge["tail"], "score": edge["relevance_score"]})
    sorted_recs = sorted(record_scores.items(), key=lambda x: -x[1])
    retrieved = []
    for rid, _ in sorted_recs[:max_records]:
        if rid in record_lookup:
            rec = record_lookup[rid]
            retrieved.append({"record_id": rid, "source_file": rec["source_file"],
                "source_row": rec.get("source_row", "unknown"),
                "text": rec["text_original"], "relevance_score": record_scores[rid],
                "supporting_edges": record_edge_map[rid][:3]})
    return retrieved

def check_domain(query, client):
    prompt = """You are a domain classifier for an aviation maintenance knowledge system.
Classify the query as IN_DOMAIN or OUT_OF_DOMAIN.
Also classify as LIKELY or UNLIKELY answerable if in-domain.
Query: "{query}"
Respond ONLY with JSON: {{"domain": "IN_DOMAIN" or "OUT_OF_DOMAIN", "answerable": "LIKELY" or "UNLIKELY", "reason": "one sentence"}}""".format(query=query)
    try:
        r = client.models.generate_content(model=GENERATION_MODEL, contents=prompt,
                                            config={"temperature": 0.0})
        raw = r.text.strip().replace("```json","").replace("```","").strip()
        return json.loads(raw)
    except:
        return {"domain": "IN_DOMAIN", "answerable": "LIKELY", "reason": "filter failed"}

def format_context(retrieved_records):
    lines = []
    for i, rec in enumerate(retrieved_records, 1):
        lines.append(f"[Record {i} | {rec['record_id']} | {rec['source_file']}]")
        lines.append(rec['text'])
        lines.append("")
    return "\n".join(lines)

print("Pipeline functions loaded.")

## 3. Pre-Generation Guardrail

**Inspired by NeMo Guardrails (Rebedea et al., 2023)**

NeMo introduces canonical form extraction: converting user utterances into
structured intent representations before policy matching. We adapt this for
knowledge graph retrieval: the canonical form of a query becomes an ideal
subschema ==> a structured specification of what entity types and relation types
SHOULD be present in the retrieved subgraph to answer the query correctly.

**Schema match score = α * structural_score + β * semantic_score**
- structural_score: fraction of required relation types present in retrieved subgraph
- semantic_score: cosine similarity between ideal entity embeddings and retrieved entity embeddings
- α=0.5, β=0.5 (equal weight, ablation in paper)

In [ ]:
IDEAL_SCHEMA_PROMPT = """You are an aviation maintenance knowledge graph expert.
Given a query, generate the IDEAL SUBSCHEMA — the minimum set of entity types
and relation types that a knowledge graph MUST contain to answer this query correctly.

Available relation types:
CAUSES, RESULTS_IN, PART_OF, LOCATED_IN, REQUIRES_ACTION,
INSPECTED_BY, REPLACED_BY, PRECEDED_BY, AFFECTS, INSTANCE_OF

Graph hub entities for context (most connected nodes):
{hub_entities}

Query: "{query}"

Generate the ideal subschema as JSON:
{{
  "canonical_form": "structured restatement of query intent in 1 sentence",
  "required_entity_types": ["list of entity concept types needed, e.g. 'hydraulic component', 'failure cause'"],
  "required_relations": ["list of relation types from the available set that must be present"],
  "ideal_triplet_patterns": [
    {{"head_type": "type", "relation": "RELATION", "tail_type": "type"}}
  ],
  "minimum_confidence_threshold": 0.0 to 1.0,
  "reasoning": "why these elements are needed"
}}
Respond ONLY with the JSON object."""
def generate_ideal_schema(query, client, graph_schema):
    hub_entities = [h["entity"] for h in graph_schema["hub_entities"][:20]]
    prompt = IDEAL_SCHEMA_PROMPT.format(
        query=query,
        hub_entities=", ".join(hub_entities)
    )
    for attempt in range(3):
        try:
            r = client.models.generate_content(
                model=GENERATION_MODEL, contents=prompt,
                config={"temperature": 0.1}
            )
            raw = r.text.strip().replace("```json","").replace("```","").strip()
            return safe_parse_json(raw)
        except Exception as e:
            if "503" in str(e):
                wait = 30 * (attempt + 1)
                print(f"  503 ideal schema, waiting {wait}s (attempt {attempt+1}/3)")
                time.sleep(wait)
                continue
            print(f"  Ideal schema error: {e}")
            return None
    return None
def compute_schema_match(ideal_schema, scored_edges, retrieved_records,
                          entity_embeddings, client, alpha=0.5, beta=0.5):
    """
    Compute match score between ideal subschema and retrieved subgraph.
    Returns: overall_score, structural_score, semantic_score, breakdown
    """
    if ideal_schema is None:
        return 0.5, 0.5, 0.5, {}  # neutral score if generation failed

    # structural score
    #te defined fraction of required relation types present in retrieved subgraph
    required_relations = set(ideal_schema.get("required_relations", []))
    retrieved_relations = set(e["relation"] for e in scored_edges)

    if required_relations:
        relations_found   = required_relations & retrieved_relations
        structural_score  = len(relations_found) / len(required_relations)
        missing_relations = required_relations - retrieved_relations
    else:
        structural_score  = 0.5
        relations_found   = set()
        missing_relations = set()

    ideal_patterns = ideal_schema.get("ideal_triplet_patterns", [])
    pattern_scores = []
    for pattern in ideal_patterns:
        req_rel  = pattern.get("relation", "")
        #here we chekcing if any retrieved edge matches this relation type
        matched = any(e["relation"] == req_rel for e in scored_edges)
        pattern_scores.append(1.0 if matched else 0.0)

    if pattern_scores:
        pattern_score    = np.mean(pattern_scores)
        structural_score = 0.5 * structural_score + 0.5 * pattern_score

    # semantic score
    # Embed required entity types and compare to retrieved entity embeddings
    required_entity_types = ideal_schema.get("required_entity_types", [])
    retrieved_entities    = list(set(
        [e["head"] for e in scored_edges] + [e["tail"] for e in scored_edges]
    ))[:50]  # cap for speed

    semantic_score = 0.5  # default
    if required_entity_types and retrieved_entities:
        try:
            # Embed required entity type descriptions
            req_result = client.models.embed_content(
                model=EMBEDDING_MODEL, contents=required_entity_types
            )
            req_embs = np.array([e.values for e in req_result.embeddings], dtype=np.float32)
            req_embs = req_embs / (np.linalg.norm(req_embs, axis=1, keepdims=True) + 1e-9)

            # Get embeddings for retrieved entities
            ret_embs_list = [entity_embeddings.get(e) for e in retrieved_entities
                             if entity_embeddings.get(e) is not None]
            if ret_embs_list:
                ret_embs = np.array(ret_embs_list, dtype=np.float32)
                ret_embs = ret_embs / (np.linalg.norm(ret_embs, axis=1, keepdims=True) + 1e-9)

                # For each required type, find max similarity to any retrieved entity
                sim_matrix = req_embs @ ret_embs.T  # (n_required, n_retrieved)
                max_sims   = sim_matrix.max(axis=1)  # best match per required type
                semantic_score = float(np.mean(max_sims))
        except Exception as e:
            print(f"  Semantic scoring error: {e}")

    #overall score
    overall_score = alpha * structural_score + beta * semantic_score

    breakdown = {
        "structural_score":  round(structural_score, 4),
        "semantic_score":    round(semantic_score,   4),
        "overall_score":     round(overall_score,    4),
        "required_relations":    list(required_relations),
        "relations_found":       list(relations_found),
        "missing_relations":     list(missing_relations),
        "required_entity_types": required_entity_types,
        "pattern_match_rate":    round(np.mean(pattern_scores), 4) if pattern_scores else None
    }

    return overall_score, structural_score, semantic_score, breakdown


# Test pre-generation guardrail
test_query  = "What causes hydraulic leaks in landing gear?"
print(f"Query: {test_query}")
ideal_schema = generate_ideal_schema(test_query, client, graph_schema)
print(f"\nIdeal schema generated:")
print(json.dumps(ideal_schema, indent=2))

In [ ]:
# Test schema match computation
query_emb    = embed_query(test_query, client)
seed_nodes   = find_seed_nodes(query_emb, entity_matrix_norm, entity_list, G)
sub_edges, _ = expand_subgraph(seed_nodes, G, hops=2)
scored_edges = score_edges_semantic(sub_edges, query_emb, entity_embeddings,
                                     relation_embeddings, top_n=30)
retrieved    = retrieve_records(scored_edges, record_lookup)

overall, structural, semantic, breakdown = compute_schema_match(
    ideal_schema, scored_edges, retrieved, entity_embeddings, client
)

print(f"Schema match results:")
print(f"  Overall score:    {overall:.4f}")
print(f"  Structural score: {structural:.4f}")
print(f"  Semantic score:   {semantic:.4f}")
print(f"  Required relations:  {breakdown['required_relations']}")
print(f"  Relations found:     {breakdown['relations_found']}")
print(f"  Missing relations:   {breakdown['missing_relations']}")
print(f"  Pattern match rate:  {breakdown['pattern_match_rate']}")

## Post-Generation Guardrail NLI grounding + graph trace

In [ ]:
POST_GUARDRAIL_PROMPT = """You are a factual grounding verifier for an aviation maintenance QA system.

QUESTION: {query}

RETRIEVED SOURCE RECORDS:
{context}

GENERATED ANSWER:
{answer}

GRAPH RETRIEVAL TRACE (edges used to retrieve records):
{trace_summary}

Evaluate the answer on three dimensions:

1. GROUNDING: Is every factual claim in the answer supported by the source records?
   - Check each claim against the records
   - Identify any claims NOT found in the records (potential hallucinations)

2. ATTRIBUTION: Can each key claim be traced to a specific record ID?
   - List which record supports which claim

3. COMPLETENESS: Does the answer address the question given what the records contain?
   - If records are insufficient, does the answer correctly say so?

Respond ONLY with JSON:
{{
  "grounding_score": 0.0 to 1.0,
  "attribution_score": 0.0 to 1.0,
  "completeness_score": 0.0 to 1.0,
  "overall_post_score": 0.0 to 1.0,
  "hallucinated_claims": ["list of claims not supported by records"],
  "well_grounded_claims": ["list of claims well supported"],
  "attribution_map": {{"claim summary": "record_id"}},
  "verdict": "PASS" or "FAIL" or "PARTIAL",
  "reasoning": "brief explanation"
}}"""

def post_generation_guardrail(query, answer, retrieved_records,
                               scored_edges, client):
    context = format_context(retrieved_records)
    trace_lines = []
    for e in scored_edges[:10]:
        trace_lines.append(
            f"{e['head']} | {e['relation']} | {e['tail']} "
            f"(score={e['relevance_score']:.3f}, "
            f"records={e['record_ids'][:3]})"
        )
    trace_summary = "\n".join(trace_lines)
    prompt = POST_GUARDRAIL_PROMPT.format(
        query=query, context=context[:3000],
        answer=answer, trace_summary=trace_summary
    )
    for attempt in range(3):
        try:
            r = client.models.generate_content(
                model=GENERATION_MODEL, contents=prompt,
                config={"temperature": 0.0}
            )
            raw = r.text.strip().replace("```json","").replace("```","").strip()
            return safe_parse_json(raw)
        except Exception as e:
            if "503" in str(e):
                wait = 30 * (attempt + 1)
                print(f"  503 post-guardrail, waiting {wait}s (attempt {attempt+1}/3)")
                time.sleep(wait)
                continue
            print(f"  Post-guardrail error: {e}")
            break
    return {"verdict": "ERROR", "overall_post_score": 0.5,
            "grounding_score": 0.5, "attribution_score": 0.5,
            "completeness_score": 0.5, "hallucinated_claims": [],
            "well_grounded_claims": [], "attribution_map": {}}


## Full guarded pipeline

In [ ]:
CONFIDENCE_THRESHOLD = 0.45  # below this: abstain or flag

GENERATION_PROMPT = """You are an aviation maintenance expert assistant.
Answer the question based ONLY on the provided maintenance records.
If the records do not contain sufficient information, say so explicitly.
Be specific and reference which records support your answer.

QUESTION: {query}

RETRIEVED MAINTENANCE RECORDS:
{context}

ANSWER:"""

def guarded_graphrag_query(query, client, G, entity_matrix_norm, entity_list,
                            entity_to_idx, entity_embeddings, relation_embeddings,
                            record_lookup, graph_schema,
                            top_k_seeds=10, hops=2, top_n_edges=30,
                            max_records=15, confidence_threshold=CONFIDENCE_THRESHOLD):
    """
    Full GraphRAG pipeline with both guardrails.
    Returns: answer, full_trace (including guardrail scores)
    """
    trace = {"query": query, "steps": {}, "guardrails": {}}

    # 1 : Domain filter
    domain_check = check_domain(query, client)
    trace["steps"]["domain_filter"] = domain_check
    if domain_check["domain"] == "OUT_OF_DOMAIN":
        trace["steps"]["abstained"]  = True
        trace["steps"]["abstain_reason"] = "out_of_domain"
        return (f"Out of domain: {domain_check['reason']}", trace)

    # 2 : Embed + seed nodes
    query_emb  = embed_query(query, client)
    seed_nodes = find_seed_nodes(query_emb, entity_matrix_norm, entity_list, G,
                                  top_k=top_k_seeds)
    trace["steps"]["seed_nodes"] = seed_nodes

    # 3 : Expand + score
    sub_edges, visited = expand_subgraph(seed_nodes, G, hops=hops)
    scored_edges = score_edges_semantic(sub_edges, query_emb, entity_embeddings,
                                         relation_embeddings, top_n=top_n_edges)
    trace["steps"]["subgraph"] = {"nodes": len(visited), "edges": len(sub_edges)}
    trace["steps"]["scored_edges"] = [{"head": e["head"], "relation": e["relation"],
        "tail": e["tail"], "score": e["relevance_score"]} for e in scored_edges[:10]]

    #  4 : Retrieve records
    retrieved = retrieve_records(scored_edges, record_lookup, max_records=max_records)
    trace["steps"]["retrieved_records"] = [{"record_id": r["record_id"],
        "source_file": r["source_file"], "source_row": r["source_row"],
        "relevance_score": r["relevance_score"]} for r in retrieved]
####### PRE-GENERATION GUARDRAIL#################################################
  
    ideal_schema = generate_ideal_schema(query, client, graph_schema)
    overall_score, structural_score, semantic_score, schema_breakdown = \
        compute_schema_match(ideal_schema, scored_edges, retrieved,
                              entity_embeddings, client)

    trace["guardrails"]["pre_generation"] = {
        "ideal_schema":       ideal_schema,
        "schema_match":       schema_breakdown,
        "overall_score":      overall_score,
        "structural_score":   structural_score,
        "semantic_score":     semantic_score,
        "threshold":          confidence_threshold,
        "passed":             overall_score >= confidence_threshold
    }

    #abstain if schema match below threshold
    if overall_score < confidence_threshold:
        searched_nodes = [s["node"] for s in seed_nodes[:5]]
        missing_rels   = schema_breakdown.get("missing_relations", [])
        abstain_msg = (
            f"The knowledge graph does not contain sufficient information to answer "
            f"this query confidently (schema match score: {overall_score:.2f}, "
            f"threshold: {confidence_threshold}). "
            f"Searched around: {searched_nodes}. "
            f"Missing graph patterns: {missing_rels}."
        )
        trace["steps"]["abstained"]    = True
        trace["steps"]["abstain_reason"] = "low_schema_match"
        return abstain_msg, trace

    # 5 : Generate answer if > threshold
    if not retrieved:
        trace["steps"]["abstained"]    = True
        trace["steps"]["abstain_reason"] = "no_records_retrieved"
        return "No relevant records found.", trace

    context = format_context(retrieved)
    prompt  = GENERATION_PROMPT.format(query=query, context=context)
    r       = client.models.generate_content(model=GENERATION_MODEL, contents=prompt,
                                              config={"temperature": 0.2})
    answer  = r.text.strip()

    ######################## POST-GENERATION GUARDRAIL#################################

    post_result = post_generation_guardrail(
        query, answer, retrieved, scored_edges, client
    )
    trace["guardrails"]["post_generation"] = post_result

    #if post-guardrail fails: flag but still return answer with warning
    if post_result.get("verdict") == "FAIL":
        hallucinated = post_result.get("hallucinated_claims", [])
        answer = (
            f"{answer}\n\n"
            f"[GUARDRAIL WARNING: {len(hallucinated)} claim(s) could not be "
            f"verified against source records: {hallucinated[:2]}]"
        )

    trace["steps"]["abstained"] = False
    return answer, trace


#testing full guarded pipeline
print(" full Guarded pipeline test")
answer, trace = guarded_graphrag_query(
    "What causes hydraulic leaks in landing gear?",
    client, G, entity_matrix_norm, entity_list, entity_to_idx,
    entity_embeddings, relation_embeddings, record_lookup, graph_schema
)
print(f"Answer: {answer[:500]}")
print(f"\nPre-guardrail score:  {trace['guardrails']['pre_generation']['overall_score']:.4f}")
print(f"  Structural:         {trace['guardrails']['pre_generation']['structural_score']:.4f}")
print(f"  Semantic:           {trace['guardrails']['pre_generation']['semantic_score']:.4f}")
print(f"  Passed threshold:   {trace['guardrails']['pre_generation']['passed']}")
if 'post_generation' in trace['guardrails']:
    pg = trace['guardrails']['post_generation']
    print(f"\nPost-guardrail:")
    print(f"  Grounding score:    {pg.get('grounding_score', 'N/A')}")
    print(f"  Attribution score:  {pg.get('attribution_score', 'N/A')}")
    print(f"  Verdict:            {pg.get('verdict', 'N/A')}")
    print(f"  Hallucinated:       {pg.get('hallucinated_claims', [])}")

## Expand benchmark to 100 questions

In [ ]:
BENCHMARK_GEN_PROMPT = """You are building a QA benchmark for an aviation maintenance knowledge system.
The system has a knowledge graph built from FAA incident records and maintenance logs.

Key topics in the graph: {hub_entities}
Relation types: CAUSES, RESULTS_IN, PART_OF, LOCATED_IN, REQUIRES_ACTION,
                INSPECTED_BY, REPLACED_BY, PRECEDED_BY, AFFECTS, INSTANCE_OF

Generate {n} questions of type: {qtype}

Type definitions:
- in_domain_answerable: questions about aviation maintenance that the graph CAN answer
- in_domain_unanswerable: aviation questions the graph CANNOT answer (too specific, not in records)
- multi_hop: questions requiring connecting 2+ relation types to answer

Rules:
1. Questions must be diverse — cover different components, failure types, actions
2. No duplicates with existing questions: {existing}
3. Questions should be natural language, not overly technical
4. For unanswerable: must be in-domain but require info not in FAA incident records

Respond ONLY with a JSON array of strings (the questions):
["question 1", "question 2", ...]"""

def generate_benchmark_questions(qtype, n, existing_questions, client, graph_schema):
    hub_entities = [h["entity"] for h in graph_schema["hub_entities"][:30]]
    existing_str = "; ".join(existing_questions[:10])
    prompt = BENCHMARK_GEN_PROMPT.format(
        qtype=qtype, n=n,
        hub_entities=", ".join(hub_entities),
        existing=existing_str
    )
    try:
        r = client.models.generate_content(
            model=GENERATION_MODEL, contents=prompt,
            config={"temperature": 0.7}  # higher temp for diversity
        )
        raw = r.text.strip().replace("```json","").replace("```","").strip()
        questions = json.loads(raw)
        return [q for q in questions if isinstance(q, str) and len(q) > 10]
    except Exception as e:
        print(f"  Generation error: {e}")
        return []


#generate additional questions per type
expanded_benchmark = {
    "in_domain_answerable":   list(qa_benchmark["in_domain_answerable"]),
    "in_domain_unanswerable": list(qa_benchmark["in_domain_unanswerable"]),
    "out_of_domain":          list(qa_benchmark["out_of_domain"]),
    "multi_hop":              list(qa_benchmark["multi_hop"])
}

targets = {
    "in_domain_answerable":   22,  # need 22 more to reach 30
    "in_domain_unanswerable": 11,  # need 11 more to reach 15
    "multi_hop":              12,  # need 12 more to reach 15
}

for qtype, n_needed in targets.items():
    print(f"Generating {n_needed} more {qtype} questions...")
    existing = [q["question"] for q in expanded_benchmark[qtype]]
    new_qs   = generate_benchmark_questions(qtype, n_needed, existing, client, graph_schema)

    for q in new_qs:
        if qtype == "in_domain_unanswerable":
            expanded_benchmark[qtype].append(
                {"question": q, "expected_behavior": "abstain",
                 "reason": "auto-generated unanswerable"})
        elif qtype == "multi_hop":
            expanded_benchmark[qtype].append(
                {"question": q, "difficulty": "hard",
                 "note": "auto-generated multi-hop"})
        else:
            expanded_benchmark[qtype].append(
                {"question": q, "difficulty": "auto"})

    print(f"  {qtype}: {len(expanded_benchmark[qtype])} questions total")
    time.sleep(1)

total = sum(len(v) for v in expanded_benchmark.values())
print(f"\nExpanded benchmark total: {total} questions")

with open("./output/qa_benchmark_expanded.json", "w") as f:
    json.dump(expanded_benchmark, f, indent=2)


## run full guarded evaluation on expanded benchmark

In [ ]:
GUARDED_CHECKPOINT = "./output/guarded_eval_checkpoint.json"

# Resume from checkpoint if exists
if os.path.exists(GUARDED_CHECKPOINT):
    with open(GUARDED_CHECKPOINT) as f:
        guarded_results = json.load(f)
    print(f"Resuming from checkpoint:")
    for qt, res in guarded_results.items():
        print(f"  {qt}: {len(res)} done")
else:
    guarded_results = {qt: [] for qt in expanded_benchmark}

def run_guarded_evaluation_checkpointed(benchmark, client, G, entity_matrix_norm,
                                         entity_list, entity_to_idx, entity_embeddings,
                                         relation_embeddings, record_lookup, graph_schema,
                                         checkpoint_file, checkpoint_every=5):
    for qtype, questions in benchmark.items():
        already_done = {r["question"] for r in guarded_results.get(qtype, [])}
        remaining    = [q for q in questions
                        if (q["question"] if isinstance(q, dict) else q) not in already_done]

        if not remaining:
            print(f"  {qtype}: already complete ({len(guarded_results[qtype])} questions)")
            continue

        print(f"\n {qtype.upper()} ({len(remaining)} remaining / {len(questions)} total)")

        for i, q_data in enumerate(tqdm(remaining, desc=qtype)):
            query  = q_data["question"] if isinstance(q_data, dict) else q_data
            answer, trace = guarded_graphrag_query(
                query, client, G, entity_matrix_norm, entity_list, entity_to_idx,
                entity_embeddings, relation_embeddings, record_lookup, graph_schema
            )
            pre_g  = trace["guardrails"].get("pre_generation", {})
            post_g = trace["guardrails"].get("post_generation", {})

            result = {
                "question":               query,
                "answer":                 answer,
                "abstained":              trace["steps"].get("abstained", False),
                "abstain_reason":         trace["steps"].get("abstain_reason", None),
                "domain_detected":        trace["steps"]["domain_filter"]["domain"],
                "records_retrieved":      len(trace["steps"].get("retrieved_records", [])),
                "pre_guardrail_score":    pre_g.get("overall_score"),
                "pre_structural_score":   pre_g.get("structural_score"),
                "pre_semantic_score":     pre_g.get("semantic_score"),
                "pre_guardrail_passed":   pre_g.get("passed"),
                "post_grounding_score":   post_g.get("grounding_score"),
                "post_attribution_score": post_g.get("attribution_score"),
                "post_verdict":           post_g.get("verdict"),
                "hallucinated_claims":    post_g.get("hallucinated_claims", []),
            }

            # Correctness flags
            if qtype == "out_of_domain":
                result["domain_correct"]     = result["domain_detected"] == "OUT_OF_DOMAIN"
                result["abstention_correct"] = result["domain_detected"] == "OUT_OF_DOMAIN"
            elif qtype == "in_domain_unanswerable":
                result["domain_correct"]     = result["domain_detected"] == "IN_DOMAIN"
                result["abstention_correct"] = result["abstained"]
            else:
                result["domain_correct"]     = result["domain_detected"] == "IN_DOMAIN"
                result["abstention_correct"] = not result["abstained"]

            guarded_results.setdefault(qtype, []).append(result)
            time.sleep(2.0)

            # Checkpoint every N questions
            if (i + 1) % checkpoint_every == 0:
                with open(checkpoint_file, "w") as f:
                    json.dump(guarded_results, f, indent=2, default=str)
                tqdm.write(f"  Checkpoint saved: {sum(len(v) for v in guarded_results.values())} total done")

    # Final save
    with open(checkpoint_file, "w") as f:
        json.dump(guarded_results, f, indent=2, default=str)
    with open("./output/eval_results_guarded.json", "w") as f:
        json.dump(guarded_results, f, indent=2, default=str)
    print("\nSaved to ./output/eval_results_guarded.json")
    return guarded_results


print("Running checkpointed guarded evaluation...")
guarded_results = run_guarded_evaluation_checkpointed(
    expanded_benchmark, client, G, entity_matrix_norm, entity_list, entity_to_idx,
    entity_embeddings, relation_embeddings, record_lookup, graph_schema,
    GUARDED_CHECKPOINT, checkpoint_every=5
)

## Comparison: baseline vs guarded metrics

In [ ]:
print(" BASELINE vs GUARDED COMPARISON")
print(f"{'Metric':<40} {'Baseline':>10} {'Guarded':>10}")
print("-" * 62)

for qtype in ["in_domain_answerable", "in_domain_unanswerable",
               "out_of_domain", "multi_hop"]:
    g_results = guarded_results.get(qtype, [])
    b_results = baseline_results.get(qtype, [])
    if not g_results:
        continue

    n_g = len(g_results)
    n_b = len(b_results)

    g_domain    = sum(1 for r in g_results if r.get("domain_correct", False)) / n_g
    g_abstain   = sum(1 for r in g_results if r.get("abstention_correct", False)) / n_g
    b_domain    = sum(1 for r in b_results if r.get("domain_correct", False)) / max(n_b, 1)
    b_abstain   = sum(1 for r in b_results if r.get("abstention_correct", False)) / max(n_b, 1)

    # post-guardrail scores (guarded only)
    post_scores = [r["post_grounding_score"] for r in g_results
                   if r.get("post_grounding_score") is not None]
    avg_grounding = np.mean(post_scores) if post_scores else None

    print(f"\n[{qtype}] (baseline n={n_b}, guarded n={n_g})")
    print(f"  {'Domain accuracy':<38} {b_domain:>10.1%} {g_domain:>10.1%}")
    print(f"  {'Abstention accuracy':<38} {b_abstain:>10.1%} {g_abstain:>10.1%}")
    if avg_grounding:
        print(f"  {'Avg grounding score (guarded)':<38} {'N/A':>10} {avg_grounding:>10.3f}")

    # hallucination rate
    halluc_counts = [len(r.get("hallucinated_claims", [])) for r in g_results]
    if halluc_counts:
        print(f"  {'Avg hallucinated claims':<38} {'N/A':>10} {np.mean(halluc_counts):>10.2f}")

    # Pre-guardrail scores
    pre_scores = [r["pre_guardrail_score"] for r in g_results
                  if r.get("pre_guardrail_score") is not None]
    if pre_scores:
        print(f"  {'Avg pre-guardrail score':<38} {'N/A':>10} {np.mean(pre_scores):>10.3f}")


**fixes**

In [ ]:
#analyze pre-guardrail score distributions by type
with open("./output/eval_results_guarded.json") as f:
    guarded_results = json.load(f)

print("PRE-GUARDRAIL SCORE DISTRIBUTION")
for qtype, results in guarded_results.items():
    scores = [r["pre_guardrail_score"] for r in results 
              if r.get("pre_guardrail_score") is not None]
    if not scores:
        continue
    print(f"\n{qtype} (n={len(scores)}):")
    print(f"  Mean:   {np.mean(scores):.4f}")
    print(f"  Median: {np.median(scores):.4f}")
    print(f"  Min:    {np.min(scores):.4f}")
    print(f"  Max:    {np.max(scores):.4f}")
    print(f"  Below 0.60: {sum(1 for s in scores if s < 0.60)}/{len(scores)}")
    print(f"  Below 0.65: {sum(1 for s in scores if s < 0.65)}/{len(scores)}")
    print(f"  Below 0.70: {sum(1 for s in scores if s < 0.70)}/{len(scores)}")
    print(f"  Below 0.75: {sum(1 for s in scores if s < 0.75)}/{len(scores)}")

In [ ]:
# based on previous results ==> re-score existing results with new threshold
NEW_THRESHOLD = 0.70

print("RESULTS WITH THRESHOLD 0.70")
for qtype, results in guarded_results.items():
    if not results:
        continue
    
    # Reapply threshold to existing scores
    for r in results:
        score = r.get("pre_guardrail_score")
        if score is not None:
            r["abstained_070"] = score < NEW_THRESHOLD
        else:
            r["abstained_070"] = r.get("abstained", False)
    
    n = len(results)
    if qtype == "in_domain_unanswerable":
        correct = sum(1 for r in results if r.get("abstained_070", False))
        print(f"\n{qtype}: abstention {correct}/{n} = {correct/n:.1%}")
    elif qtype == "out_of_domain":
        correct = sum(1 for r in results if r["domain_detected"] == "OUT_OF_DOMAIN")
        print(f"\n{qtype}: rejection {correct}/{n} = {correct/n:.1%}")
    else:
        #answerable ==> should NOT abstain
        false_abstentions = sum(1 for r in results if r.get("abstained_070", False))
        print(f"\n{qtype}: false abstentions {false_abstentions}/{n} = {false_abstentions/n:.1%}")

#save updated results
with open("./output/eval_results_guarded_t070.json", "w") as f:
    json.dump(guarded_results, f, indent=2, default=str)